In [26]:
import os
import sys

# Add project root to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from pinecone import Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
from langchain_core.messages import HumanMessage, SystemMessage


# Cost for Langchain SDK usage

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.oauth2 import service_account
from backend.core.config import settings

credentials = service_account.Credentials.from_service_account_file(
    settings.GOOGLE_APPLICATION_CREDENTIALS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

llm = ChatGoogleGenerativeAI(
    model=settings.GEMINI_MODEL_NAME,  # Use the model from settings
    credentials=credentials,
    project=settings.GOOGLE_CLOUD_PROJECT,
    location=settings.GOOGLE_CLOUD_LOCATION,
    temperature=0
)

response = llm.invoke("2+2")  # should return "4"
print(response)

input_tokens = response.usage_metadata.get("input_tokens", None)
print(f"Input tokens: {input_tokens}")


TypeError: expected str, bytes or os.PathLike object, not SecretStr

In [26]:
input_tokens = response.usage_metadata.get("input_tokens", None)
print(input_tokens) # number of input tokens

3


In [28]:
output_tokens = response.usage_metadata.get("output_tokens", None)
print(output_tokens) # number of output tokens

61


 # Cost for SDK usage

In [ ]:
from google import genai
from backend.core.config import settings

model = genai.Client(
    vertexai=True,  
    project=settings.GOOGLE_CLOUD_PROJECT,
    location=settings.GOOGLE_CLOUD_LOCATION,
)   

response = model.models.generate_content(
    model=settings.GEMINI_MODEL_NAME,  # Use the model from settings
    contents="2+2",
    config=genai.types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.1
    )
)

print(response)


DefaultCredentialsError: File /home/chichi/work/ocr-extraction/shivautomation-479307-2b7eac169823.json was not found.

In [ ]:
input_tokens = response.usage_metadata.prompt_token_count
print(input_tokens) # number of input tokens
# for gemini 2.5 lite For Input (text, image, video)	$0.1 per 1M tokens
# for gemini 2.5 flash For Input (text, image, video)	$0.3 per 1M tokens

input_tokens_price = (input_tokens / 1000000) * 0.1 
print(f"{input_tokens_price:.8f}")

3
0.00000030


In [ ]:
candidates_tokens = response.usage_metadata.candidates_token_count
thoughts_tokens = response.usage_metadata.thoughts_token_count
total_output_tokens = candidates_tokens + thoughts_tokens
print(total_output_tokens) # number of output tokens
# for gemini 2.5 lite For Output (text, image, video)	$0.4 per 1M tokens
# for gemini 2.5 flash For Output (text, image, video)	$2.5 per 1M tokens

output_tokens_price = (total_output_tokens / 1000000) * 0.4 
print(f"{output_tokens_price:.8f}")

24
0.00000960


### Logic per model of Gemini

# Hybrid Retriever

In [10]:
from pinecone import Pinecone
from langchain_community.retrievers import PineconeHybridSearchRetriever
# from sentence_transformers import SentenceTransformer
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder

In [ ]:
from backend.core.config import settings
embeddings = HuggingFaceEmbeddings(model_name=settings.EMBEDDING_MODEL)

In [12]:
bm25_encoder = BM25Encoder().default()

In [ ]:
from backend.core.config import settings

pc = Pinecone(settings.PINECONE_API_KEY)
index = pc.Index(settings.PINECONE_INDEX_NAME) 
namespaces = ["raw_material", "auxiliary_raw_material", "electrical"]

total_fetched_result = []

for namespace in namespaces:
    query = "gl side slit"
    retriever = PineconeHybridSearchRetriever(
                        embeddings= embeddings,
                        sparse_encoder=bm25_encoder,
                        index=index,
                        namespace=namespace,
                        top_k= 25,
                        alpha=0.1,
                        text_key="text",
                    )
    result = retriever.invoke(query)
    print(result)

    for doc in result:
        fetched_result = [doc.page_content]
        total_fetched_result.extend(fetched_result)
        

print(len(total_fetched_result))
print(total_fetched_result)

[Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02792', 'UoMGroupEntry': 1, 'score': 0.0244930983}, page_content='MS WIRE 8 MM'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM2795', 'UoMGroupEntry': 1, 'score': 0.0241150856}, page_content='MS WIRE 5.50 MM'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02786', 'UoMGroupEntry': 1, 'score': 0.0229854584}, page_content='Light Scrap'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02783', 'UoMGroupEntry': 1, 'score': 0.0191881657}, page_content='Dehati Scrap'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02787', 'UoMGroupEntry': 1, 'score': 0.0190014839}, page_content='Light/Tina/Burada'), Document(metadata={'InventoryUoMEntry': 1, 'ItemCode': 'RM02781', 'UoMGroupEntry': 1, 'score': 0.0163851976}, page_content='Burada Scrap'), Document(metadata={'InventoryUoMEntry': 10, 'ItemCode': 'RM02794', 'UoMGroupEntry': 9, 'score': 0.00866216421}, page_content='ENAMEL RED PAINT'), Documen